In [103]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,id,chirp,label
0,25758,@user bihday greg t,0
1,12137,looks like le monsieur bob approves of the new...,0
2,29855,"#cavycorners, hooded_cavy #f4f rss: hooded_cavy",0
3,20936,filming a documentary today in eindhoven! #d...,0
4,21638,@user #petitsfilousfirsts @user britmumspics ...,0


In [104]:
# subtasks

subtask1 = df_test.chirp.map(lambda x: len(x))
subtask2 = df_test.chirp.map(lambda x: x.count(r'#'))

df_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.id,
    "answer": subtask1
})

df_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.id,
    "answer": subtask2
})

subtasks = pd.concat([df_1, df_2])

subtasks

,subtaskID,datapointID,answer
0,1,12413,38
1,1,9312,55
2,1,12110,72
3,1,28747,131
4,1,7658,104
...,...,...,...
3192,2,16717,0
3193,2,22887,4
3194,2,3300,1
3195,2,17599,0


In [105]:
df_test.chirp[3]

"@user hey #congress how about instead of sending out thoughts &amp; prayers u do something it's ur fucking job!   #orlando ð\x9f\x87ºð\x9f\x87¸"

In [106]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'\d+', " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    return ' '.join(tokens)

preprocess(df_test.chirp[3])

'@us hey # congress instead send thought & amp ; prayer u ur fucking job ! # orlando ð\x9f\x87ºð\x9f\x87¸'

In [107]:
df_train.chirp = df_train.chirp.apply(preprocess)
df_test.chirp = df_test.chirp.apply(preprocess)

df_train.head()

,id,chirp,label
0,25758,@user bihday greg t,0
1,12137,look like le monsieur bob approve new shag rug...,0
2,29855,"# cavycorner , hooded_cavy # f f rss : hooded_...",0
3,20936,film documentary today eindhoven ! # documenta...,0
4,21638,@us # petitsfilousfirst @user britmumspic # sn...,0


In [108]:
features = 'chirp'
target = 'label'

X = df_train[features]
y = df_train[target]
X_final = df_test[features]

from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer(lowercase=True,ngram_range=(3,5), analyzer='char')

X = vect.fit_transform(X)
X_final = vect.transform(X_final)

X

<28765x341959 sparse matrix of type '<class 'numpy.float64'>'
	with 4754921 stored elements in Compressed Sparse Row format>

In [109]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score

model = LogisticRegression()

cross_val_score(model, X, y, cv=2, scoring='roc_auc')

array([0.95350693, 0.95463205])

In [110]:
model.fit(X, y)
y_preds = model.predict_proba(X_final)[:, 1]

In [111]:
df_preds = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": df_test.id,
    "answer": y_preds
})

df_preds.head()

,subtaskID,datapointID,answer
0,3,12413,0.006989
1,3,9312,0.007018
2,3,12110,0.020511
3,3,28747,0.033865
4,3,7658,0.012179


In [112]:
submission = pd.concat([subtasks, df_preds])

submission

,subtaskID,datapointID,answer
0,1,12413,38.000000
1,1,9312,55.000000
2,1,12110,72.000000
3,1,28747,131.000000
4,1,7658,104.000000
...,...,...,...
3192,3,16717,0.004144
3193,3,22887,0.017583
3194,3,3300,0.045599
3195,3,17599,0.003181


In [113]:
submission.to_csv("submission.csv", index=False)